In [2]:
#Exception chaining - raise from:

In [3]:
def load_config(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError as e:
        raise RuntimeError(f"Config missing: {path}") from e
        # The original error is preserved as __cause__

try:
    load_config("missing.cfg")
except RuntimeError as e:
    print(e)                    # Config missing: missing.cfg
    print(e.__cause__)          # [Errno 2] No such file...

Config missing: missing.cfg
[Errno 2] No such file or directory: 'missing.cfg'


In [4]:
#Suppressing exceptions - raise from None:

In [5]:
try:
    d = {}
    val = d["key"]
except KeyError:
    raise ValueError("key not found") from None
    # Hides the original KeyError — cleaner error message

ValueError: key not found

In [6]:
#Exception groups (Python 3.11+):

In [7]:
# Collect multiple errors then raise together:
errors = []
for item in data:
    try:
        process(item)
    except ValueError as e:
        errors.append(e)
if errors:
    raise ExceptionGroup("validation failed", errors)

NameError: name 'data' is not defined

In [9]:
#Context managers with with - the right way to handle resources:

In [10]:
# Without with (risky — file may not close if exception):
f = open("file.txt")
data = f.read()    # if this crashes, f never closes!
f.close()

# With with (safe — always closes):
with open("file.txt") as f:
    data = f.read()   # f.close() called automatically

FileNotFoundError: [Errno 2] No such file or directory: 'file.txt'

In [11]:
#contextlib.suppress - suppress specific exceptions:

In [12]:
from contextlib import suppress

with suppress(FileNotFoundError):
    os.remove("temp.txt")   # if not found, just skip
# No try/except needed!

NameError: name 'os' is not defined

In [13]:
#contextlib.contextmanager — build your own context manager:

In [14]:
from contextlib import contextmanager

@contextmanager
def managed_resource(name):
    print(f"Acquiring {name}")
    try:
        yield name        # code inside `with` runs here
    except Exception as e:
        print(f"Error in {name}: {e}")
        raise
    finally:
        print(f"Releasing {name}")

with managed_resource("database") as db:
    print(f"Using {db}")
# Acquiring database
# Using database
# Releasing database

Acquiring database
Using database
Releasing database


In [15]:
#Retry pattern - most important real-world pattern:

In [16]:
import time

def with_retry(func, max_attempts=3, delay=1, exceptions=(Exception,)):
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except exceptions as e:
            if attempt == max_attempts:
                raise
            print(f"Attempt {attempt} failed: {e}. Retrying in {delay}s...")
            time.sleep(delay)

In [17]:
#Logging exceptions - production pattern:

In [18]:
import traceback

try:
    risky_operation()
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")
    print(traceback.format_exc())   # full stack trace as string

ERROR: NameError: name 'risky_operation' is not defined
Traceback (most recent call last):
  File "C:\Users\91973\AppData\Local\Temp\ipykernel_13624\2160858034.py", line 4, in <module>
    risky_operation()
    ^^^^^^^^^^^^^^^
NameError: name 'risky_operation' is not defined



In [19]:
#Transaction pattern - all or nothing:

In [20]:
def transaction(operations, rollback):
    completed = []
    try:
        for op in operations:
            op()
            completed.append(op)
    except Exception as e:
        print(f"Failed at step {len(completed)+1}: {e}")
        print(f"Rolling back {len(completed)} operations...")
        for op in reversed(completed):
            rollback(op)
        raise

In [21]:
#examples

In [22]:
#finally and else mastery

In [27]:
fake_fs = {
    "data.txt": "Hello World",
    "config.json": "{\"key\":\"value\"}",
    "empty.txt": ""
}

def safe_open_and_read(filename, encoding="utf-8"):
    try:
        # Force TypeError when filename is None
        if filename in fake_fs:
            content = fake_fs[filename]
        else:
            raise KeyError

    except KeyError:
        print(f"File not found: {filename}")

    except Exception as e:
        print(f"Unexpected error: {e}")

    else:
        print(f"Read {len(content)} chars from {filename}")
        return content

    finally:
        print(f"Operation complete for: {filename}")


# Test cases
safe_open_and_read("data.txt")
print()

safe_open_and_read("empty.txt")
print()

safe_open_and_read("missing.txt")
print()

safe_open_and_read(None)

Read 11 chars from data.txt
Operation complete for: data.txt

Read 0 chars from empty.txt
Operation complete for: empty.txt

File not found: missing.txt
Operation complete for: missing.txt

File not found: None
Operation complete for: None


In [23]:
#Exception chaining

In [28]:
# Custom exceptions
class ConfigError(Exception):
    pass


class AppInitError(Exception):
    pass


# Step 1: Parse JSON
def parse_json(text):
    if text.startswith("{") and text.endswith("}"):
        return {
            "parsed": True,
            "data": text
        }

    raise ValueError("Invalid JSON format")


# Step 2: Load config
def load_config(source):
    try:
        return parse_json(source)

    except ValueError as e:
        raise ConfigError("Failed to load config") from e


# Step 3: Initialize app
def init_app(config_source):
    try:
        config = load_config(config_source)
        print("App started successfully!")
        return config

    except ConfigError as e:
        raise AppInitError("App failed to start") from e


# -----------------------------
# VALID TEST
# -----------------------------
print("VALID CONFIG:")
init_app('{"debug": true}')

print("\n" + "-" * 40 + "\n")


# -----------------------------
# INVALID TEST
# -----------------------------
print("INVALID CONFIG:")

try:
    init_app("invalid config text")

except AppInitError as e:
    print(f"{type(e).__name__}: {e}")

    cause1 = e.__cause__
    print(f"Caused by: {type(cause1).__name__}: {cause1}")

    cause2 = cause1.__cause__
    print(f"Caused by: {type(cause2).__name__}: {cause2}")

VALID CONFIG:
App started successfully!

----------------------------------------

INVALID CONFIG:
AppInitError: App failed to start
Caused by: ConfigError: Failed to load config
Caused by: ValueError: Invalid JSON format


In [24]:
#Retry decorator

In [29]:
import time


def retry(max_attempts=3, delay=0, exceptions=(Exception,), on_failure=None):
    # Decorator factory
    def decorator(func):

        def wrapper(*args, **kwargs):
            last_error = None

            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)

                except exceptions as e:
                    last_error = e

                    print(
                        f"Attempt {attempt}/{max_attempts} failed: {e}"
                    )

                    # Delay between retries
                    if delay > 0:
                        time.sleep(delay)

            # All attempts failed
            if on_failure:
                on_failure(last_error)
            else:
                raise last_error

        return wrapper

    return decorator


# ---------------------------------------------------
# TEST 1: flaky function
# ---------------------------------------------------

counter = [0]


@retry(max_attempts=3)
def flaky_counter():
    counter[0] += 1

    if counter[0] < 3:
        raise ValueError("not ready yet")

    return 42


result = flaky_counter()
print(f"Success after retries: {result}")


# ---------------------------------------------------
# TEST 2: always fails
# ---------------------------------------------------

@retry(
    max_attempts=3,
    on_failure=lambda e: print(f"Giving up: {e}")
)
def always_fails():
    raise RuntimeError("always broken")


always_fails()

Attempt 1/3 failed: not ready yet
Attempt 2/3 failed: not ready yet
Success after retries: 42
Attempt 1/3 failed: always broken
Attempt 2/3 failed: always broken
Attempt 3/3 failed: always broken
Giving up: always broken


In [25]:
#Context manager from scratch

In [30]:
import time


# ---------------------------------------------------
# TIMER CONTEXT MANAGER
# ---------------------------------------------------

class Timer:

    def __enter__(self):
        self.start_time = time.time()
        print("Timer started")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):

        end_time = time.time()
        elapsed = end_time - self.start_time

        if exc_type is not None:
            print(f"Error during timed block: {exc_val}")

        print(f"Elapsed: ~{elapsed:.1f}s")

        # False = do NOT suppress exceptions
        return False


# ---------------------------------------------------
# TRANSACTION CONTEXT MANAGER
# ---------------------------------------------------

class Transaction:

    def __init__(self, name):
        self.name = name
        self.operations = []

    def __enter__(self):
        print(f"BEGIN {self.name}")
        return self

    def add_operation(self, op_name):
        self.operations.append(op_name)

    def __exit__(self, exc_type, exc_val, exc_tb):

        # SUCCESS
        if exc_type is None:
            print(f"COMMIT {self.name}")

            if self.operations:
                print(
                    "Operations:",
                    ", ".join(self.operations)
                )

        # FAILURE
        else:
            print(f"ROLLBACK {self.name}")

            for op in reversed(self.operations):
                print(f"Undoing: {op}")

        # False = propagate exception
        return False


# ---------------------------------------------------
# TIMER SUCCESS
# ---------------------------------------------------

with Timer():
    x = sum(range(1000))


print()


# ---------------------------------------------------
# TIMER FAILURE
# ---------------------------------------------------

try:
    with Timer():
        raise ValueError("intentional error")

except ValueError:
    pass


print()


# ---------------------------------------------------
# TRANSACTION SUCCESS
# ---------------------------------------------------

with Transaction("order_123") as tx:
    tx.add_operation("charge_card")
    tx.add_operation("update_inventory")
    tx.add_operation("send_email")


print()


# ---------------------------------------------------
# TRANSACTION FAILURE
# ---------------------------------------------------

try:
    with Transaction("order_456") as tx:
        tx.add_operation("update_inventory")
        tx.add_operation("charge_card")

        raise RuntimeError("payment failed")

except RuntimeError:
    pass

Timer started
Elapsed: ~0.0s

Timer started
Error during timed block: intentional error
Elapsed: ~0.0s

BEGIN order_123
COMMIT order_123
Operations: charge_card, update_inventory, send_email

BEGIN order_456
ROLLBACK order_456
Undoing: charge_card
Undoing: update_inventory


In [26]:
#Production error handler

In [31]:
from datetime import datetime


# ---------------------------------------------------
# CUSTOM EXCEPTION
# ---------------------------------------------------

class CircuitOpenError(Exception):
    pass


# ---------------------------------------------------
# ERROR LOGGER
# ---------------------------------------------------

class ErrorLog:

    def __init__(self):
        self.logs = []

    def log(self, error, context={}):

        entry = {
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "type": type(error).__name__,
            "message": str(error),
            "context": context
        }

        self.logs.append(entry)

    def summary(self):

        print("--- Error Summary ---")

        counts = {}

        for log in self.logs:
            err_type = log["type"]
            counts[err_type] = counts.get(err_type, 0) + 1

        for err_type, count in counts.items():
            print(f"{err_type}: {count}")

    def report(self):

        print("\n--- Full Error Report ---")

        for log in self.logs:
            print(log)


# ---------------------------------------------------
# CIRCUIT BREAKER
# ---------------------------------------------------

class CircuitBreaker:

    def __init__(self, name, failure_threshold=3, reset_after=5):

        self.name = name
        self.failure_threshold = failure_threshold
        self.reset_after = reset_after

        self.failures = 0
        self.open = False

    def call(self, func, *args, **kwargs):

        # Circuit already OPEN
        if self.failures >= self.failure_threshold:
            self.open = True
            raise CircuitOpenError(
                f"Circuit {self.name} is OPEN (skipping)"
            )

        try:
            result = func(*args, **kwargs)

            # Reset failures on success
            self.failures = 0
            return result

        except Exception:

            self.failures += 1

            # Open circuit if threshold reached
            if self.failures >= self.failure_threshold:
                self.open = True
                print(f"Circuit {self.name} OPENED")

            raise

    def status(self):

        print("\n--- Circuit Status ---")

        state = "OPEN" if self.open else "CLOSED"

        print(
            f"{self.name}: {state} ({self.failures} failures)"
        )


# ---------------------------------------------------
# FLAKY SERVICE
# ---------------------------------------------------

service_counter = [0]


def flaky_service():

    service_counter[0] += 1

    # First 2 calls succeed
    if service_counter[0] <= 2:
        return "success"

    # Afterwards fail forever
    raise ConnectionError("service down")


# ---------------------------------------------------
# TEST SYSTEM
# ---------------------------------------------------

logger = ErrorLog()

breaker = CircuitBreaker(
    "db_service",
    failure_threshold=3
)


for i in range(1, 8):

    try:
        result = breaker.call(flaky_service)

        print(f"Call {i}: {result}")

    except Exception as e:

        print(f"Call {i}: {e}")

        logger.log(
            e,
            context={"call_number": i}
        )


# ---------------------------------------------------
# RESULTS
# ---------------------------------------------------

logger.summary()

breaker.status()

Call 1: success
Call 2: success
Call 3: service down
Call 4: service down
Circuit db_service OPENED
Call 5: service down
Call 6: Circuit db_service is OPEN (skipping)
Call 7: Circuit db_service is OPEN (skipping)
--- Error Summary ---
ConnectionError: 3
CircuitOpenError: 2

--- Circuit Status ---
db_service: OPEN (3 failures)
